# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [21]:
!pip install langchain_community langchain_experimental langchain_openai

In [22]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [23]:
from typing import Optional
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.tools import DuckDuckGoSearchRun
import faiss

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)

#### Using DuckDuckGoSearchRun(ddgs) replacing eith SerpAPIWrapper

In [24]:
%pip install -q ddgs

Note: you may need to restart the kernel to use updated packages.


In [25]:
from ddgs import DDGS

def web_search(query: str) -> str:
    """Searches the web and returns the top results as text."""
    try:
        results = DDGS().text(query, max_results=5)
        if not results:
            return "No results found. Try a different search query."
        return "\n\n".join(f"{r['title']}: {r['body']}" for r in results)
    except Exception as e:
        return f"Search error: {e}. Try again or use a different query."

print(web_search("San Francisco weather forecast"))

February 14, 2026 San Francisco weather forecast: February 14, 2026, San Francisco weather forecast documented the predicted atmospheric conditions for that date in San Francisco, California, amid an emerging active weather pattern in the Bay Area. Forecasts anticipated a transition toward unsettle…

10-Day Weather Forecast for San Francisco, California - The Weather ...: Be prepared with the most accurate 10-day forecast for San Francisco, California with highs, lows, chance of precipitation from The Weather Channel and Weather.com

Weather Forecast and Conditions for San Francisco, California - The ...: Today's and tonight's San Francisco, California weather forecast, weather conditions and Doppler radar from The Weather Channel and Weather.com

San Francisco, CA Weather Forecast | AccuWeather: San Francisco, CA Weather Forecast, with current conditions, wind, air quality, and what to expect for the next 3 days.

San Francisco Weather Forecast: San Francisco Weather Forecast. Providi

In [26]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [27]:
%pip install -q langchain_community langchain_experimental langchain_openai faiss-cpu duckduckgo-search

Note: you may need to restart the kernel to use updated packages.


In [28]:


todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=llm, prompt=todo_prompt)
#search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=web_search,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [29]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)

tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [30]:
OBJECTIVE = "Write a weather report for SF today"

In [31]:
# Logging of LLMChains
verbose = False

baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=3,
)

In [32]:
#baby_agi({"objective": OBJECTIVE})
baby_agi.invoke({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Question: What is the weather report for San Francisco today?
Thought: I need to find the current weather conditions for San Francisco to provide an accurate report.
Action: Search
Action Input: "current weather San Francisco October 2023"
Observation: San Francisco October 2023 Historical Weather Data ...: It features all historical weather data series we have available, including the San Francisco temperature history for October 2023. You can drill down from year to month and even day level reports by clicking on the graphs.

Weather in San Francisco in October 2023: San Francisco Weather Forecast for October 2023 is based on statistical data. Wind direction chart for San Francisco (daytime), based on October 2023 wind statistics. Average daytime values for October 2023.

San Francisco 2023 Past Weather (California, United States ...: It features all historical weather data series we have available, including the San Francisco temperature history for 2023. You can drill down from yea

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Thought: I need to find the current weather data for San Francisco to create an accurate weather report.
Action: Search
Action Input: "current weather San Francisco"
Observation: Search error: No results found.. Try again or use a different query.
Thought:I need to try a different query to gather the current weather data for San Francisco.
Action: Search
Action Input: "San Francisco weather report today"
Observation: Weather Forecast and Conditions for San Francisco, California ...: Today’s and tonight’s San Francisco, California weather forecast, weather conditions and Doppler radar from The Weather Channel and Weather.com

Weather forecast and conditions for San Francisco, California ...: Today’s and tonight’s San Francisco, California, United States weather forecast, weather conditions and Doppler radar from The Weather Channel and Weather.com

San Francisco Weather Today – Hourly Forecast | Ease Weather: San Francisco weather today, hour by hour: temperature, chance of rain, wind, 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



*****TASK LIST*****

3: Analyze the gathered weather data for San Francisco.
4: Draft a weather report for San Francisco based on the analyzed data.
5: Review and edit the drafted weather report for clarity and accuracy.
6: Format the final weather report for presentation.
7: Create a summary of the key weather trends for San Francisco over the past week.
8: Research and compile information on typical weather patterns for San Francisco in October.
9: Develop a list of outdoor activities suitable for today's weather in San Francisco.
10: Generate a social media post highlighting today's weather in San Francisco.
11: Prepare a comparison of today's weather in San Francisco with another major city.

*****NEXT TASK*****

3: Analyze the gathered weather data for San Francisco.
Thought: I need to analyze the current weather data for San Francisco to provide an accurate weather report.
Action: Search
Action Input: "current weather data for San Francisco"
Observation: Current Weather Observat

{'objective': 'Write a weather report for SF today'}